In [1]:
import pandas as pd
import io
from google.colab import files
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
print("Please upload your 'yield_df.csv' file:")
uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

print(f"\nSuccessfully loaded {file_name}!")
df.head()

Please upload your 'yield_df.csv' file:


Saving yield_df.csv to yield_df.csv

Successfully loaded yield_df.csv!


,Unnamed: 0,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,0,Albania,Maize,1990,36613,1485.0,121.0,16.37
1,1,Albania,Potatoes,1990,66667,1485.0,121.0,16.37
2,2,Albania,"Rice, paddy",1990,23333,1485.0,121.0,16.37
3,3,Albania,Sorghum,1990,12500,1485.0,121.0,16.37
4,4,Albania,Soybeans,1990,7000,1485.0,121.0,16.37


In [3]:
# Drop unnecessary tracking columns like index and year
cols_to_drop = [col for col in ['Unnamed: 0', 'Year'] if col in df.columns]
df = df.drop(columns=cols_to_drop)

# Dynamically identify the yield column (the target)
target_col = [col for col in df.columns if 'yield' in col.lower()][0]

# Separate features (X) and target (y)
y = df[target_col]
X_raw = df.drop(columns=[target_col])

# Convert text-based columns into numerical format
X = pd.get_dummies(X_raw, drop_first=True)

print(f"Data ready. Total features after encoding: {X.shape[1]}")

Data ready. Total features after encoding: 112


In [4]:
# Split 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the inputs
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data successfully split and scaled!")

Data successfully split and scaled!


In [5]:
# Initialize the L2 Regularization model
ridge = Ridge()

# Provide the grid of alpha punishment levels to test
param_grid = {'alpha': [0.01, 0.1, 1, 10, 100]}

# Run GridSearchCV to find the absolute best alpha
grid_search = GridSearchCV(ridge, param_grid, cv=5)
grid_search.fit(X_train_scaled, y_train)

# The AI will print the winning parameter
print(f"GridSearch complete!")
print(f"The best punishment level (alpha) is: {grid_search.best_params_['alpha']}")

GridSearch complete!
The best punishment level (alpha) is: 10


In [6]:
# Extract the optimal model found by GridSearch
best_model = grid_search.best_estimator_

# Generate predictions on the unseen test data
y_pred = best_model.predict(X_test_scaled)

# Calculate error and accuracy
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:,.2f}")
print(f"R-squared Score: {r2:.4f}")

Mean Squared Error (MSE): 1,818,133,553.04
R-squared Score: 0.7493
